In [1]:
print("Ok")

Ok


In [1]:
import requests
from bs4 import BeautifulSoup

def fetch_pubmed_articles_with_metadata(query: str, max_results=3, use_mock_if_empty=True):
    headers = {"User-Agent": "Mozilla/5.0"}

    # Step 1: Search PubMed
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    try:
        search_response = requests.get(search_url, params=search_params, headers=headers, timeout=10).json()
        id_list = search_response["esearchresult"]["idlist"]
        print("Found PubMed IDs:", id_list)
        if not id_list:
            raise ValueError("No IDs found for this query.")

        ids = ",".join(id_list)

        # Step 2: Fetch article summaries
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {
            "db": "pubmed",
            "id": ids,
            "retmode": "xml"
        }
        fetch_response = requests.get(fetch_url, params=fetch_params, headers=headers, timeout=10)
        soup = BeautifulSoup(fetch_response.text, "lxml")
        articles_xml = soup.find_all("pubmedarticle")
        print("Articles found in XML:", len(articles_xml))

        articles_info = []
        for article, pmid in zip(articles_xml, id_list):
            title_tag = article.find("articletitle")
            abstract_tag = article.find("abstract")
            date_tag = article.find("pubdate")
            author_tags = article.find_all("author")

            # Title
            title = title_tag.get_text(strip=True) if title_tag else "No title"

            # Abstract
            abstract = abstract_tag.get_text(separator=" ", strip=True) if abstract_tag else "No abstract available"

            # Authors
            authors = []
            for author in author_tags:
                last = author.find("lastname")
                fore = author.find("forename")
                if last and fore:
                    authors.append(f"{fore.get_text()} {last.get_text()}")
                elif last:
                    authors.append(last.get_text())
            authors = authors if authors else ["No authors listed"]

            # Publication Date
            pub_date = "No date"
            if date_tag:
                year = date_tag.find("year")
                month = date_tag.find("month")
                pub_date = f"{month.get_text()} {year.get_text()}" if year and month else year.get_text() if year else "No date"

            # PubMed Article URL
            url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

            print(f"Article: {title}\n   - Authors: {authors}\n   - Date: {pub_date}\n   - URL: {url}\n")
            articles_info.append({
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "publication_date": pub_date,
                "article_url": url
            })

        if not articles_info and use_mock_if_empty:
            print("No valid articles found, returning mock data.")
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        return articles_info

    except Exception as e:
        print(f"Error during PubMed fetch: {e}")
        if use_mock_if_empty:
            return [{
                "title": "Simulated Study on Fever",
                "abstract": "This is a simulated abstract on the treatment of fever in adults.",
                "authors": ["John Doe", "Jane Smith"],
                "publication_date": "March 2024",
                "article_url": "https://pubmed.ncbi.nlm.nih.gov/12345678/"
            }]
        else:
            return [{"message": f"Error: {e}"}]


In [2]:
fetch_pubmed_articles_with_metadata("back pain")

Found PubMed IDs: ['41789369', '41789279', '41789024']
Articles found in XML: 3
Article: Health-related quality of life among patients with low back pain attending rehabilitation facilities in Bangladesh: a cross-sectional study.
   - Authors: ['Sohel Ahmed', 'Hafez Mohammad Ibrahim', 'Fahim Morshed', 'Mst Akhima Zurtte Mitu', 'Mostafa Hosen', 'Md Rasel Uddin', 'Md Amran Hossain', 'Md Selim Rana', 'Tasnuva Shamarukh Proma', 'Tofajjal Hossain', 'Mohammad Jahirul Islam']
   - Date: 2026
   - URL: https://pubmed.ncbi.nlm.nih.gov/41789369/

Article: High- and Low-Level Laser Therapy for the Treatment of Orthopedic Pain: A Systematic Review.
   - Authors: ['Gabrielly Santos Pereira', 'Joelington Dias Batista', 'Jobson Dias Batista', 'Ludimila Dias Silva', 'Josie Resende Torres da Silva', 'João Eduardo de Araújo', 'Marcelo Lourenço da Silva']
   - Date: 2025
   - URL: https://pubmed.ncbi.nlm.nih.gov/41789279/

Article: Incessant pericarditis due to lung cancer and obstructive pneumonia.
   -

/var/folders/8q/brg24f8j29xcd5_x4y87gp140000gn/T/ipykernel_13875/340134985.py:32: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(fetch_response.text, "lxml")


[{'title': 'Health-related quality of life among patients with low back pain attending rehabilitation facilities in Bangladesh: a cross-sectional study.',
  'abstract': 'Low back pain (LBP) is a serious health issue that may impact a person at some point in their life, which negatively impacts the health-related quality of life (HRQOL) of those affected people. This study aimed to investigate the HRQOL of patients with LBP in Bangladesh. A multicentre cross-sectional study was conducted between August 2024 and February 2025, including 369 patients with LBP. This study recruited both male and female individuals aged 18 to 60 years with a structured questionnaire. Descriptive statistics, as well as bivariate and multivariate analyses, were conducted to analysed the data. The significance level was set at <0.05. Widowed/divorced LBP survivors encounter significantly greater challenges in mobility (adjusted OR (aOR) 29.37, 95%\u2009CI 4.75 to 181.87) and self-care (aOR 8.30, 95%\u2009CI 1.